In [1]:
import pandas as pd

import football_predictor

In [2]:
df = pd.read_csv("../data/raw/Matches.csv")

/var/folders/28/lbx47lx91ds2yl8gz5v4g2380000gn/T/ipykernel_10479/2189308214.py:1: DtypeWarning: Columns (0: MatchTime) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/Matches.csv")


In [3]:
filtered_df = df[df['Division'] == "SP1"].copy()
filtered_df.shape

(9419, 48)

In [4]:

filtered_df['MatchDate'] = pd.to_datetime(filtered_df['MatchDate'])
filtered_df.shape


(9419, 48)

In [5]:
recent = filtered_df[filtered_df['MatchDate'] >= '2022-08-01'].copy()
recent.shape

(1551, 48)

In [6]:
recent['HomeTeam'].unique()

<StringArray>
[    'Osasuna',       'Celta',  'Valladolid',   'Barcelona',       'Cadiz',
    'Valencia',     'Almeria',  'Ath Bilbao',      'Getafe',       'Betis',
     'Espanol',     'Sevilla',    'Mallorca',  'Ath Madrid',    'Sociedad',
       'Elche',      'Girona',   'Vallecano', 'Real Madrid',  'Villarreal',
  'Las Palmas',      'Alaves',     'Granada',     'Leganes',     'Levante',
      'Oviedo',   'Santander',   'La Coruna',      'Malaga']
Length: 29, dtype: str

In [7]:
recent['FTHome'].isna().sum()

np.int64(0)

In [8]:
home_avg = recent['FTHome'].mean()
print("home avg:" , home_avg)

home avg: 1.4958091553836235


In [9]:
away_avg = recent['FTAway'].mean()
print("away avg:" , away_avg)

away avg: 1.1263700838168924


In [10]:
barca_home = recent[recent['HomeTeam'] == "Barcelona"]
barca_home_scored = barca_home['FTHome'].mean()
barca_home_attack = barca_home_scored / home_avg

print("home games:" , len(barca_home))
print("avg scored at home:" , barca_home_scored)
print("home attack strength:" , barca_home_attack)

home games: 78
avg scored at home: 2.5128205128205128
home attack strength: 1.6799071618037136


In [11]:
recent.groupby('HomeTeam')['FTHome'].mean()

HomeTeam
Alaves         1.118644
Almeria        1.342105
Ath Bilbao     1.545455
Ath Madrid     2.153846
Barcelona      2.512821
Betis          1.558442
Cadiz          0.921053
Celta          1.397436
Elche          1.230769
Espanol        1.271186
Getafe         0.974026
Girona         1.789474
Granada        1.263158
La Coruna      2.000000
Las Palmas     1.078947
Leganes        1.210526
Levante        1.550000
Malaga         1.000000
Mallorca       1.184211
Osasuna        1.384615
Oviedo         0.473684
Real Madrid    2.435897
Santander      2.500000
Sevilla        1.217949
Sociedad       1.423077
Valencia       1.282051
Valladolid     0.842105
Vallecano      1.233766
Villarreal     2.144737
Name: FTHome, dtype: float64

In [12]:
home_attack  = recent.groupby('HomeTeam')['FTHome'].mean() / home_avg
home_defense = recent.groupby('HomeTeam')['FTAway'].mean() / away_avg
away_attack  = recent.groupby('AwayTeam')['FTAway'].mean() / away_avg
away_defense = recent.groupby('AwayTeam')['FTHome'].mean() / home_avg

ratings = pd.DataFrame({
    'home_attack': home_attack,
    'home_defense': home_defense,
    'away_attack': away_attack,
    'away_defense': away_defense
})

ratings.head(29)

,home_attack,home_defense,away_attack,away_defense
Alaves,0.747852,0.917903,0.872501,1.037381
Almeria,0.897244,1.355075,0.957898,1.442627
Ath Bilbao,1.033190,0.807098,1.013011,0.874237
Ath Madrid,1.439920,0.808133,1.256767,0.711946
Barcelona,1.679907,0.648783,1.867855,0.729310
Betis,1.041872,0.887808,1.069922,0.959947
Cadiz,0.615755,0.981261,0.490631,1.161139
Celta,0.934234,1.104069,1.058540,1.045656
Elche,0.822812,1.206508,0.754637,1.337069
Espanol,0.849832,1.218855,0.994957,1.129593


In [13]:
ratings.isna().sum()

home_attack     0
home_defense    0
away_attack     0
away_defense    0
dtype: int64

In [14]:
import os
# test api key
print(bool(os.environ.get("FOOTBALL_API_KEY")))

True


In [15]:
import os, requests, json
resp = requests.get("https://api.football-data.org/v4/competitions/PD/matches",
    headers={"X-Auth-Token": os.environ.get("FOOTBALL_API_KEY")}
)
resp.raise_for_status()
data = resp.json()

with open("../data/raw/fixtures_PD.json", "w") as f:
    json.dump(data, f)

print(data["resultSet"])
print(len(data["matches"]), "matches")

{'count': 380, 'first': '2026-08-15', 'last': '2027-05-30', 'played': 41}
380 matches


In [16]:
m = data["matches"][0]
print(json.dumps(m, indent=2)[:1500])

{
  "area": {
    "id": 2224,
    "name": "Spain",
    "code": "ESP",
    "flag": "https://crests.football-data.org/760.svg"
  },
  "competition": {
    "id": 2014,
    "name": "Primera Division",
    "code": "PD",
    "type": "LEAGUE",
    "emblem": "https://crests.football-data.org/laliga.png"
  },
  "season": {
    "id": 2518,
    "startDate": "2026-08-16",
    "endDate": "2027-05-30",
    "currentMatchday": 6,
    "winner": null
  },
  "id": 564634,
  "utcDate": "2026-08-15T17:30:00Z",
  "status": "FINISHED",
  "matchday": 1,
  "stage": "REGULAR_SEASON",
  "group": null,
  "lastUpdated": "2026-09-11T00:20:30Z",
  "homeTeam": {
    "id": 263,
    "name": "Deportivo Alav\u00e9s",
    "shortName": "Alav\u00e9s",
    "tla": "ALA",
    "crest": "https://crests.football-data.org/263.png"
  },
  "awayTeam": {
    "id": 82,
    "name": "Getafe CF",
    "shortName": "Getafe",
    "tla": "GET",
    "crest": "https://crests.football-data.org/82.png"
  },
  "score": {
    "winner": "HOME_TEAM"

In [17]:
fixtures = pd.DataFrame([
    {
        "matchday": m["matchday"],
        "utc": m["utcDate"],
        "status": m["status"],
        "home": m["homeTeam"]["name"],
        "away": m["awayTeam"]["name"],
        "fh": m["score"]["fullTime"]["home"],
        "fa": m["score"]["fullTime"]["away"],
    }
    for m in data["matches"]
])
fixtures["utc"] = pd.to_datetime(fixtures["utc"])
fixtures.shape, fixtures["status"].value_counts()

((380, 7),
 status
 SCHEDULED    300
 FINISHED      41
 TIMED         39
 Name: count, dtype: int64)

In [18]:
api_names = sorted(set(fixtures["home"]) | set(fixtures["away"]))
print(len(api_names))
for n in api_names:
    print(repr(n))

print(sorted(ratings.index.tolist()))

20
'Athletic Club'
'CA Osasuna'
'Club Atlético de Madrid'
'Deportivo Alavés'
'Elche CF'
'FC Barcelona'
'Getafe CF'
'Levante UD'
'Málaga CF'
'RC Celta de Vigo'
'RC Deportivo La Coruña'
'RCD Espanyol de Barcelona'
'Rayo Vallecano de Madrid'
'Real Betis Balompié'
'Real Madrid CF'
'Real Racing Club de Santander'
'Real Sociedad de Fútbol'
'Sevilla FC'
'Valencia CF'
'Villarreal CF'
['Alaves', 'Almeria', 'Ath Bilbao', 'Ath Madrid', 'Barcelona', 'Betis', 'Cadiz', 'Celta', 'Elche', 'Espanol', 'Getafe', 'Girona', 'Granada', 'La Coruna', 'Las Palmas', 'Leganes', 'Levante', 'Malaga', 'Mallorca', 'Osasuna', 'Oviedo', 'Real Madrid', 'Santander', 'Sevilla', 'Sociedad', 'Valencia', 'Valladolid', 'Vallecano', 'Villarreal']


In [19]:
NAME_MAP = {
    "Athletic Club": "Ath Bilbao",
    "CA Osasuna": "Osasuna",
    "Club Atlético de Madrid": "Ath Madrid",
    "Deportivo Alavés": "Alaves",
    "Elche CF": "Elche",
    "FC Barcelona": "Barcelona",
    "Getafe CF": "Getafe",
    "Levante UD": "Levante",
    "Málaga CF": "Malaga",
    "RC Celta de Vigo": "Celta",
    "RC Deportivo La Coruña": "La Coruna",
    "Real Betis Balompié": "Betis",
    "Real Madrid CF": "Real Madrid",
    "Real Sociedad de Fútbol": "Sociedad",
    "Rayo Vallecano de Madrid": "Vallecano",
    "RCD Espanyol de Barcelona": "Espanol",
    "Real Racing Club de Santander": "Santander",
    "Sevilla FC": "Sevilla",
    "Valencia CF": "Valencia",
    "Villarreal CF": "Villarreal",
}

fixtures["home_r"] = fixtures["home"].map(NAME_MAP)
fixtures["away_r"] = fixtures["away"].map(NAME_MAP)


unmapped = (
    set(fixtures.loc[fixtures["home_r"].isna(), "home"]) |
    set(fixtures.loc[fixtures["away_r"].isna(), "away"])
)
print("unmapped teams:", unmapped)
assert not (set(NAME_MAP.values()) - set(ratings.index))
assert fixtures[["home_r", "away_r"]].notna().all().all(), "unmapped team"

unmapped teams: set()


In [20]:
from collections import Counter

played = Counter(recent["HomeTeam"]) + Counter(recent["AwayTeam"])
for t in sorted(set(NAME_MAP.values())):
    n = played.get(t, 0)
    flag = "⚠️" if n < 10 else ""
    print(f"{t:14} {played.get(t,0)}")

Alaves         117
Ath Bilbao     155
Ath Madrid     155
Barcelona      155
Betis          155
Celta          156
Elche          79
Espanol        117
Getafe         155
La Coruna      3
Levante        41
Malaga         3
Osasuna        155
Real Madrid    155
Santander      3
Sevilla        155
Sociedad       156
Valencia       155
Vallecano      155
Villarreal     155


In [21]:
k = 10
n = pd.Series(played).reindex(ratings.index).fillna(0)    # match count per rated team
w = n / (n + k)                                          # weighted on raw rating 0-1 because of minimal data
ratings_adj = ratings.mul(w, axis=0).add(1 - w, axis=0)

print(ratings_adj.loc[["Barcelona", "Malaga", "Santander", "La Coruna"]])
print(ratings_adj.describe())

           home_attack  home_defense  away_attack  away_defense
Barcelona     1.638701      0.670068     1.815258      0.745716
Malaga        0.923508      0.974109     0.769231      1.232062
Santander     1.154924      1.178988     0.769231      0.923508
La Coruna     1.077785      0.974109     0.974109      0.923508
       home_attack  home_defense  away_attack  away_defense
count    29.000000     29.000000    29.000000     29.000000
mean      0.962424      1.037801     0.945391      1.052567
std       0.275120      0.186356     0.260357      0.221752
min       0.459034      0.670068     0.549860      0.680467
25%       0.825499      0.894607     0.776973      0.923508
50%       0.909192      1.035413     0.882540      1.033571
75%       1.039334      1.183305     1.048181      1.119389
max       1.638701      1.392077     1.815258      1.608821


In [22]:
def expected_goals(home, away, R = ratings_adj):
    eh = home_avg * R.loc[home, "home_attack"] * R.loc[away, "away_defense"]
    ea = away_avg * R.loc[away, "away_attack"] * R.loc[home, "home_defense"]
    return eh, ea

print(expected_goals("Barcelona", "Real Madrid"))
print(expected_goals("Real Madrid", "Barcelona"))

(np.float64(1.6679500294272036), np.float64(1.1656872815596115))
(np.float64(1.7739999609402295), np.float64(1.457505153585822))


In [23]:
eg = fixtures.apply(lambda r: expected_goals(r["home_r"], r["away_r"]), axis=1, result_type="expand")
fixtures[["xg_home", "xg_away"]] = eg
fixtures.loc[fixtures["status"] == "FINISHED", ["home", "away", "matchday", "xg_home", "xg_away"]].head(12)

,home,away,matchday,xg_home,xg_away
0,Deportivo Alavés,Getafe CF,1,1.021922,0.820120
1,Sevilla FC,Rayo Vallecano de Madrid,1,1.148561,0.997998
2,Real Racing Club de Santander,Villarreal CF,1,1.629093,1.594581
3,RCD Espanyol de Barcelona,Levante UD,1,1.440213,1.231396
4,RC Deportivo La Coruña,Elche CF,1,2.094513,0.858242
5,Club Atlético de Madrid,Málaga CF,1,2.604537,0.710272
6,Rayo Vallecano de Madrid,Deportivo Alavés,2,1.292683,1.029269
7,Real Betis Balompié,Real Sociedad de Fútbol,2,1.333341,1.009202
8,Athletic Club,Sevilla FC,2,1.653438,0.994876
9,Valencia CF,RC Celta de Vigo,2,1.350570,1.050387


In [24]:
import numpy as np
from scipy.stats import poisson

def score_grid(lh, la, max_goals=10):
    # Return a grid of probabilities for each scoreline (home, away) given expected goals lh and la
    h = poisson.pmf(np.arange(max_goals + 1), lh)
    a = poisson.pmf(np.arange(max_goals + 1), la)
    grid = np.outer(h, a)
    return grid / grid.sum()



In [25]:
def summarize(grid):
    home_win = np.tril(grid, -1).sum()  # cells where i > j where home scored more
    draw = np.trace(grid)              # cells where i == j
    away_win = np.triu(grid, 1).sum()   # cells where i < j where away scored more
    i, j = np.unravel_index(grid.argmax(), grid.shape) # most likely scoreline
    return {"Score" : (i, j), "Home" : home_win, "Draw" : draw, "Away" : away_win}

In [26]:
g = score_grid(*expected_goals("Barcelona", "Real Madrid"))
summarize(g)

{'Score': (np.int64(1), np.int64(1)),
 'Home': np.float64(0.491012711283205),
 'Draw': np.float64(0.24227474574374475),
 'Away': np.float64(0.26671254297305036)}

In [27]:
def fit_ratings(matches, k=10, half_life = None, ref_date = None):
    m = matches.copy()
    if half_life:
        ref = ref_date if ref_date is not None else m["MatchDate"].max()
        age = (ref - m["MatchDate"]).dt.days.clip(lower=0)
        m["w"] = 0.5 ** (age / half_life)
    else:
        m["w"] = 1.0

    def wavg(group_col, val_col):
        num = (m["w"] * m[val_col]).groupby(m[group_col]).sum()
        den = m["w"].groupby(m[group_col]).sum()
        return num / den

    ha = (m["w"] * m["FTHome"]).sum() / m["w"].sum()
    aa = (m["w"] * m["FTAway"]).sum() / m["w"].sum()

    raw = pd.DataFrame({
        "home_attack" : wavg("HomeTeam", "FTHome") / ha,
        "home_defense" : wavg("HomeTeam", "FTAway") / aa,
        "away_attack" : wavg("AwayTeam", "FTAway") / aa,
        "away_defense" : wavg("AwayTeam", "FTHome") / ha,
    })

    n = (m["w"].groupby(m["HomeTeam"]).sum().add(m["w"].groupby(m["AwayTeam"]).sum(), fill_value=0).reindex(raw.index).fillna(0))
    w = n / (n + k)
    adj = raw.mul(w, axis = 0).add(1 - w, axis = 0)
    return {"home_avg": ha, "away_avg": aa, "ratings": adj}


In [28]:
def predict_hda(model, home, away, max_goals = 10):
    R, ha, aa = model["ratings"], model["home_avg"], model["away_avg"]
    if home not in R.index or away not in R.index:
        return None
    lh = ha * R.loc[home, "home_attack"] * R.loc[away, "away_defense"]
    la = aa * R.loc[away, "away_attack"] * R.loc[home, "home_defense"]
    g = score_grid(lh, la, max_goals)
    return np.tril(g, -1).sum(), np.trace(g), np.triu(g,1).sum()

In [29]:
m_all = fit_ratings(recent)
print(predict_hda(m_all, "Barcelona", "Real Madrid"))
print(m_all["home_avg"], m_all["away_avg"])

(np.float64(0.491012711283205), np.float64(0.24227474574374475), np.float64(0.26671254297305036))
1.4958091553836235 1.1263700838168924


In [30]:
rs = recent.sort_values("MatchDate")
train, test = rs.iloc[:-150], rs.iloc[-150:]
model = fit_ratings(train)
print(len(train), len(test), test["MatchDate"].min().date(), "->", test["MatchDate"].max().date())


1401 150 2026-03-07 -> 2026-09-03


In [31]:
OUT = {"H" : 0, "D" : 1, "A" : 2}

recs = []
for _, r in test.iterrows():
    p = predict_hda(model, r["HomeTeam"], r["AwayTeam"])
    if p is None:
        continue            # team not in training slice
    actual = r["FTResult"]
    recs.append({"p_H": p[0], "p_D": p[1], "p_A": p[2], "actual": actual})
res = pd.DataFrame(recs)
print(len(res), "of", len(test), "scored")
res.head()

142 of 150 scored


,p_H,p_D,p_A,actual
0,0.573115,0.234549,0.192336,H
1,0.291206,0.244395,0.464399,A
2,0.509071,0.264746,0.226183,D
3,0.256229,0.237687,0.506083,D
4,0.741670,0.152359,0.105971,H


In [32]:
def log_loss(res):
    p = res[["p_H", "p_D", "p_A"]].to_numpy()
    inx = res["actual"].map(OUT).to_numpy()
    return -np.log(p[np.arange(len(p)), inx]).mean()

def rps(res):
    p = res[["p_H", "p_D", "p_A"]].to_numpy()
    oh = np.eye(3)[res["actual"].map(OUT).to_numpy()]
    cp, co = np.cumsum(p, axis=1), np.cumsum(oh, axis=1)
    return (((cp - co) ** 2).sum(axis=1) / 2).mean()

print("model log loss", round(log_loss(res), 4), "RPS", round(rps(res), 4))

model log loss 1.01 RPS 0.2156


In [33]:
recs = []
for _, r in test.iterrows():
    p = predict_hda(model, r["HomeTeam"], r["AwayTeam"])
    if p is None:
        continue
    recs.append({
        "p_H": p[0],
        "p_D": p[1],
        "p_A": p[2],
        "actual": r["FTResult"],
        "OddHome": r["OddHome"],
        "OddDraw": r["OddDraw"],
        "OddAway": r["OddAway"],
        "HomeElo": r["HomeElo"],
        "AwayElo": r["AwayElo"],
    })

res = pd.DataFrame(recs).dropna(
    subset = ["OddHome", "OddDraw", "OddAway", "HomeElo", "AwayElo"]
).reset_index(drop=True)
len(res)

142

In [34]:
# Bookmaker - implied probability is 1/odd, normalization is needed because of margin
inv = 1 / res[["OddHome", "OddDraw", "OddAway"]].to_numpy()
book = inv / inv.sum(axis=1, keepdims=True)
res_book = res.assign(p_H = book[:,0], p_D = book[:,1], p_A = book[:,2])


In [35]:
# Elo - logistic on rating difference, with 400 rating points = 10x odds
d = (res["HomeElo"] - res["AwayElo"]).to_numpy() + 65 # elo home advantage
ph = 1 / (1 + 10 ** (-d / 400))
draw = .26
res_elo = res.assign(p_D = draw, p_H = (1 - draw) * ph, p_A = (1 - draw) * (1 - ph))

In [36]:
base = train["FTResult"].value_counts(normalize=True)
res_base = res.assign(p_H = base["H"], p_D = base["D"], p_A = base["A"])

for name, r in [("model", res), ("base rate", res_base), ("Elo", res_elo), ("bookmaker",res_book)]:
    print(f"{name:11} log loss {log_loss(r):.4f} RPS {rps(r):.4f}")

model       log loss 1.0100 RPS 0.2156
base rate   log loss 1.0284 RPS 0.2208
Elo         log loss 1.0088 RPS 0.2122
bookmaker   log loss 0.9747 RPS 0.2025


In [37]:
def evaluate(model):
    recs = []
    for _, r in test.iterrows():
        p = predict_hda(model, r["HomeTeam"], r["AwayTeam"])
        if p is None:
            continue
        recs.append({
            "p_H": p[0],
            "p_D": p[1],
            "p_A": p[2],
            "actual": r["FTResult"],
        })
    rr = pd.DataFrame(recs)
    return log_loss(rr), rps(rr)

ref = train["MatchDate"].max()
for hl in [None, 60, 90, 120, 150, 180, 240, 365, 500]:      # half-life in days - for this set bottoms at 30-45 days then rises again
    ll, r = evaluate(fit_ratings(train, half_life = hl, ref_date = ref))
    print(f"half_life {str(hl):>5} log loss {ll:.4f} RPS {r:.4f}")
    # Likely to pick hl = 75 showing roughly last 8 match days dominate the ratings, but not too much to overfit to recent results


half_life  None log loss 1.0100 RPS 0.2156
half_life    60 log loss 0.9798 RPS 0.2045
half_life    90 log loss 0.9772 RPS 0.2040
half_life   120 log loss 0.9776 RPS 0.2043
half_life   150 log loss 0.9793 RPS 0.2050
half_life   180 log loss 0.9813 RPS 0.2057
half_life   240 log loss 0.9852 RPS 0.2070
half_life   365 log loss 0.9913 RPS 0.2091
half_life   500 log loss 0.9954 RPS 0.2105


In [38]:
ref = train["MatchDate"].max()
model = fit_ratings(train, k=10, half_life=90, ref_date = ref)

In [39]:
BEST_HL = 90
for k in [3, 6, 10, 12, 16, 20, 25, 35, 50]:
    ll, r = evaluate(fit_ratings(train, k = k, half_life = BEST_HL, ref_date = ref))
    print(f"k {k:>3} log loss {ll:.4f} RPS {r:.4f}")


k   3 log loss 0.9823 RPS 0.2064
k   6 log loss 0.9780 RPS 0.2046
k  10 log loss 0.9772 RPS 0.2040
k  12 log loss 0.9777 RPS 0.2040
k  16 log loss 0.9796 RPS 0.2044
k  20 log loss 0.9819 RPS 0.2050
k  25 log loss 0.9848 RPS 0.2059
k  35 log loss 0.9900 RPS 0.2076
k  50 log loss 0.9961 RPS 0.2096


In [40]:
full_model = fit_ratings(recent, k = 10, half_life = 75, ref_date = recent["MatchDate"].max())

In [41]:
# prediction wrapper that gives scoreline
def predict_full(model, home, away, max_goals = 10):
    R, ha, aa = model["ratings"], model["home_avg"], model["away_avg"]
    if home not in R.index or away not in R.index:
        return None
    lh = ha * R.loc[home, "home_attack"] * R.loc[away, "away_defense"]
    la = aa * R.loc[away, "away_attack"] * R.loc[home, "home_defense"]
    g = score_grid(lh, la, max_goals)
    i, j = np.unravel_index(g.argmax(), g.shape)
    return {"score" : (int(i), int(j)), "H": np.tril(g, - 1).sum(), "D": np.trace(g), "A": np.triu(g, 1).sum(), "xg": (round(lh, 2), round(la, 2))}

In [42]:
barca = fixtures[(fixtures["home_r"] == "Barcelona") | (fixtures["away_r"] == "Barcelona")].copy()

THIN = {"Malaga", "Santander", "La Coruna"}  # teams with few matches in the training set
rows = []
for _, m in barca.sort_values("matchday").iterrows():
    p = predict_full(full_model, m["home_r"], m["away_r"])
    rows.append({
        "matchday": m["matchday"],
        "fixture": f"{m['home']} vs {m['away']}",
        "status": m["status"],
        "pred": f'{round(p["xg"][0])}-{round(p["xg"][1])}',
        "H/D/A" : f'{p["H"]:.2f}/{p["D"]:.2f}/{p["A"]:.2f}',
        "actual": "" if pd.isna(m["fh"]) else f'{int(m["fh"])}-{int(m["fa"])}',
        "flag": "⚠️" if m["home_r"] in THIN or m["away_r"] in THIN else "",
    })
barca_pred = pd.DataFrame(rows)
barca_pred

,matchday,fixture,status,pred,H/D/A,actual,flag
0,1,FC Barcelona vs Athletic Club,FINISHED,2-1,0.69/0.18/0.14,2-0,
1,2,Elche CF vs FC Barcelona,FINISHED,1-2,0.17/0.19/0.64,0-5,
2,3,FC Barcelona vs Rayo Vallecano de Madrid,FINISHED,3-1,0.74/0.15/0.11,5-2,
3,4,Valencia CF vs FC Barcelona,FINISHED,1-2,0.24/0.24/0.51,0-5,
4,5,Levante UD vs FC Barcelona,TIMED,2-2,0.35/0.22/0.43,,
5,6,FC Barcelona vs Real Racing Club de Santander,TIMED,2-1,0.69/0.19/0.12,,⚠️
6,7,Sevilla FC vs FC Barcelona,TIMED,1-2,0.22/0.22/0.56,,
7,8,FC Barcelona vs Getafe CF,TIMED,2-1,0.70/0.19/0.11,,
8,9,Real Betis Balompié vs FC Barcelona,SCHEDULED,1-1,0.34/0.26/0.40,,
9,10,FC Barcelona vs Real Madrid CF,SCHEDULED,2-1,0.55/0.23/0.22,,


In [43]:
p = predict_full(full_model, "Barcelona", "Ath Bilbao")
print(p)
print("ratings:", full_model["ratings"].loc["Barcelona"].to_dict())
print("avgs:" , full_model["home_avg"], full_model["away_avg"])

{'score': (2, 0), 'H': np.float64(0.6853500035808295), 'D': np.float64(0.17674745482414483), 'A': np.float64(0.13790254159502563), 'xg': (np.float64(2.38), np.float64(0.98))}
ratings: {'home_attack': 1.4087303622572591, 'home_defense': 0.8420025756341211, 'away_attack': 1.5417902249598734, 'away_defense': 0.8074683947059089}
avgs: 1.6457343115510765 1.1257089847479456


In [44]:
KEEP = ["Division", "MatchDate", "HomeTeam", "AwayTeam", "FTHome", "FTAway",
          "FTResult", "HomeElo", "AwayElo", "OddHome", "OddDraw", "OddAway"]
recent[KEEP].to_csv("../data/laliga.csv", index=False)

In [45]:
# self-contained re-tune + benchmark against the fixed fit_ratings
import numpy as np, pandas as pd
from football_predictor.data import load_matches
from football_predictor.model import fit_ratings, predict

rs = load_matches().sort_values("MatchDate").reset_index(drop=True)
train, test = rs.iloc[:-150], rs.iloc[-150:]
ref = train["MatchDate"].max()
OUT = {"H": 0, "D": 1, "A": 2}

def frame(model, rows):
    recs = []
    for _, r in rows.iterrows():
        p = predict(model, r["HomeTeam"], r["AwayTeam"])
        if p is None:
            continue
        recs.append({"p_H": p["H"], "p_D": p["D"], "p_A": p["A"], "actual": r["FTResult"],
                     "OddHome": r["OddHome"], "OddDraw": r["OddDraw"], "OddAway": r["OddAway"],
                     "HomeElo": r["HomeElo"], "AwayElo": r["AwayElo"]})
    return pd.DataFrame(recs)

def log_loss(res):
    p = res[["p_H", "p_D", "p_A"]].to_numpy()
    ix = res["actual"].map(OUT).to_numpy()
    return -np.log(p[np.arange(len(p)), ix]).mean()

def rps(res):
    p = res[["p_H", "p_D", "p_A"]].to_numpy()
    oh = np.eye(3)[res["actual"].map(OUT).to_numpy()]
    cp, co = np.cumsum(p, axis=1), np.cumsum(oh, axis=1)
    return (((cp - co) ** 2).sum(axis=1) / 2).mean()

print("half_life sweep (k=10)")
for hl in [None, 45, 60, 75, 90, 120, 150, 200]:
    r = frame(fit_ratings(train, k=10, half_life=hl, ref_date=ref), test)
    print(f"  hl {str(hl):>4}  logloss {log_loss(r):.4f}  RPS {rps(r):.4f}  (n={len(r)})")

print("\nk sweep (half_life=90)")
for k in [3, 6, 10, 12, 16, 20, 25]:
    r = frame(fit_ratings(train, k=k, half_life=90, ref_date=ref), test)
    print(f"  k {k:>3}  logloss {log_loss(r):.4f}  RPS {rps(r):.4f}")

res = frame(fit_ratings(train, k=10, half_life=90, ref_date=ref), test).dropna(
    subset=["OddHome", "OddDraw", "OddAway", "HomeElo", "AwayElo"]).reset_index(drop=True)
base = train["FTResult"].value_counts(normalize=True)
res_base = res.assign(p_H=base["H"], p_D=base["D"], p_A=base["A"])
inv = 1 / res[["OddHome", "OddDraw", "OddAway"]].to_numpy()
book = inv / inv.sum(axis=1, keepdims=True)
res_book = res.assign(p_H=book[:, 0], p_D=book[:, 1], p_A=book[:, 2])
d = (res["HomeElo"] - res["AwayElo"]).to_numpy() + 65
ph = 1 / (1 + 10 ** (-d / 400))
res_elo = res.assign(p_D=0.26, p_H=(1 - 0.26) * ph, p_A=(1 - 0.26) * (1 - ph))

print(f"\nbenchmark (n={len(res)})")
for name, rr in [("model", res), ("base rate", res_base), ("Elo", res_elo), ("bookmaker", res_book)]:
    print(f"  {name:11} logloss {log_loss(rr):.4f}  RPS {rps(rr):.4f}")


half_life sweep (k=10)
  hl None  logloss 1.0100  RPS 0.2156  (n=142)
  hl   45  logloss 0.9836  RPS 0.2054  (n=142)
  hl   60  logloss 0.9798  RPS 0.2045  (n=142)
  hl   75  logloss 0.9779  RPS 0.2041  (n=142)
  hl   90  logloss 0.9772  RPS 0.2040  (n=142)
  hl  120  logloss 0.9776  RPS 0.2043  (n=142)
  hl  150  logloss 0.9793  RPS 0.2050  (n=142)
  hl  200  logloss 0.9827  RPS 0.2062  (n=142)

k sweep (half_life=90)
  k   3  logloss 0.9823  RPS 0.2064
  k   6  logloss 0.9780  RPS 0.2046
  k  10  logloss 0.9772  RPS 0.2040
  k  12  logloss 0.9777  RPS 0.2040
  k  16  logloss 0.9796  RPS 0.2044
  k  20  logloss 0.9819  RPS 0.2050
  k  25  logloss 0.9848  RPS 0.2059

benchmark (n=142)
  model       logloss 0.9772  RPS 0.2040
  base rate   logloss 1.0284  RPS 0.2208
  Elo         logloss 1.0088  RPS 0.2122
  bookmaker   logloss 0.9747  RPS 0.2025


In [46]:
from football_predictor.evaluate import log_loss, rps

In [47]:
from football_predictor.evaluate import split, predictions
from football_predictor.model import fit_ratings, FINAL
from football_predictor.data import load_matches

matches = load_matches()
train, test = split(matches)
f = predictions(fit_ratings(train, **FINAL, ref_date=train["MatchDate"].max()), test)
print(len(f), "of", len(test))
f.head()

142 of 150


,p_H,p_D,p_A,actual
0,0.641531,0.207209,0.151260,D
1,0.327350,0.271356,0.401294,D
2,0.607076,0.218563,0.174361,H
3,0.304003,0.226515,0.469482,A
4,0.667708,0.182859,0.149433,H


In [48]:
from football_predictor.evaluate import evaluate

ref = train["MatchDate"].max()
evaluate(fit_ratings(train, **FINAL, ref_date = ref), test)

{'n': 142,
 'log_loss': np.float64(0.9771726548897322),
 'rps': np.float64(0.20396985875998924)}

In [49]:
from football_predictor.evaluate import benchmark

ref = train["MatchDate"].max()
benchmark(fit_ratings(train, **FINAL, ref_date = ref), train, test)

,n,log_loss,rps
method,,,
model,142,0.977173,0.203970
base rate,142,1.028376,0.220831
Elo + 65,142,1.008750,0.212206
bookmaker,142,0.974689,0.202520


In [50]:
from football_predictor.fixtures import fetch_fixtures, FIXTURES_JSON

print(FIXTURES_JSON, FIXTURES_JSON.exists())

/Users/alexkiss/PycharmProjects/FootballPredictor/data/raw/fixtures_PD.json True


In [51]:
from football_predictor.fixtures import load_fixtures

fx = load_fixtures(refresh = False)
fx.shape, fx["status"].value_counts()

((380, 9),
 status
 SCHEDULED    300
 FINISHED      41
 TIMED         39
 Name: count, dtype: int64)

In [52]:
from football_predictor.fixtures import thin_teams
thin_teams(recent)

{'La Coruna', 'Malaga', 'Santander'}

In [53]:
from football_predictor.fixtures import load_fixtures, fixtures_to_matches
fx = load_fixtures()
fm = fixtures_to_matches(fx)
fm.shape, fm.head()

((41, 6),
             MatchDate   HomeTeam    AwayTeam  FTHome  FTAway FTResult
 0 2026-08-15 17:30:00     Alaves      Getafe     3.0     0.0        H
 1 2026-08-15 19:30:00    Sevilla   Vallecano     2.0     1.0        H
 2 2026-08-16 15:00:00  Santander  Villarreal     2.0     2.0        D
 3 2026-08-16 17:00:00    Espanol     Levante     3.0     0.0        H
 4 2026-08-17 19:00:00  La Coruna       Elche     1.0     1.0        D)